In [16]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.power as smp
import statsmodels.stats.proportion as smprop

In [17]:
from pathlib import Path

# Load and combine all label CSVs into one DataFrame.
csv_files = []
for data_dir in (Path("Data"), Path("Project_work/Data")):
    csv_files = sorted(data_dir.glob("labels_*.csv"))
    if csv_files:
        break

if not csv_files:
    raise FileNotFoundError("Could not find any labels_*.csv files in Data/ or Project_work/Data/")

DF = pd.concat((pd.read_csv(csv_file) for csv_file in csv_files), ignore_index=True)


In [18]:
DF.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   file          1200 non-null   str    
 1   prompt_class  1200 non-null   str    
 2   seed          1200 non-null   int64  
 3   class         1200 non-null   str    
 4   confidence    1200 non-null   float64
dtypes: float64(1), int64(1), str(3)
memory usage: 47.0 KB


We clean the data by removing unnesecary columns and renaming columns for clarity

In [ ]:
# Data preprocessing
DF = DF.drop(columns=['file', 'seed'])
DF['female'] = (DF['female'] == 'W').astype(int)
DF = DF.rename(columns={'prompt_class': 'adjective'})
DF['adjective'] = DF['adjective'].astype('category')

We want to remove low confidence images we set the limit at 0.85 to remove a maximum of 5 iamges per adjective

In [20]:
# Count rows per adjective with confidence below 0.85
summary_low_conf = DF[DF['confidence'] < 0.85].groupby('adjective').size()
print(summary_low_conf)

# remove rows with confidence below 0.85
DF = DF[DF['confidence'] >= 0.85]

adjective
assertive             1
bossy                 5
control               3
ditzy                 2
emotional             5
hysterical            4
irrational            3
passionate            2
sexually_confident    1
silly                 5
dtype: int64


In [21]:
Path("data").mkdir(exist_ok=True)
DF.to_csv(Path("Data") / "dataset_cleaned.csv", index=False)